In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("../data/raw/accessguru")
df = pd.read_csv(DATA / "Original_full_data_new.csv")

print(df.shape)
print(df.columns.tolist())
df.head(3)

(3524, 17)
['id', 'web_URL_id', 'domain_category', 'web_URL', 'scrape_status', 'html_file_name', 'html_file_path', 'violation_count', 'violation_name', 'violation_score', 'violation_description', 'violation_description_url', 'affected_html_elements', 'violation_category', 'violation_impact', 'wcag_reference', 'supplementary_information']


,id,web_URL_id,domain_category,web_URL,scrape_status,html_file_name,html_file_path,violation_count,violation_name,violation_score,violation_description,violation_description_url,affected_html_elements,violation_category,violation_impact,wcag_reference,supplementary_information
0,700_0,700,Government and Public Services,https://www.usa.gov/about-the-us,scraped,www_usa_gov_about_the_us.html,/content/workspace/FullPipeline/html_pages_asy...,4,color-contrast-enhanced,4,Ensures the contrast between foreground and ba...,https://dequeuniversity.com/rules/axe/4.4/colo...,"<a href=""/buy-from-government"">How to buy from...",Layout,serious,['1.4.6 Contrast (Enhanced)'],"{'fgColor': '#00bde3', 'bgColor': '#112f4e', '..."
1,700_1,700,Government and Public Services,https://www.usa.gov/about-the-us,scraped,www_usa_gov_about_the_us.html,/content/workspace/FullPipeline/html_pages_asy...,4,landmark-banner-is-top-level,3,Ensures the banner landmark is at top level,https://dequeuniversity.com/rules/axe/4.4/land...,"<div class=""usa-banner__header"" role=""banner"">",Syntax,moderate,['1.3.1 Info and Relationships'],"<div class=""usa-banner__header"" role=""banner"">..."
2,700_2,700,Government and Public Services,https://www.usa.gov/about-the-us,scraped,www_usa_gov_about_the_us.html,/content/workspace/FullPipeline/html_pages_asy...,4,landmark-no-duplicate-banner,3,Ensures the document has at most one banner la...,https://dequeuniversity.com/rules/axe/4.4/land...,"<header class=""usa-header usa-header--extended...",Syntax,moderate,['1.3.1 Info and Relationships'],11 <header> or role='banner' elements found:\n...


In [2]:
from urllib.parse import urlparse

df["domain"] = df["web_URL"].map(lambda u: urlparse(u).netloc.replace("www.", ""))

print("violations (rows):", len(df))
print("pages (unique URLs):", df["web_URL"].nunique())
print("websites (unique domains):", df["domain"].nunique())
print("violation types:", df["violation_name"].nunique())

violations (rows): 3524
pages (unique URLs): 588
websites (unique domains): 464
violation types: 94


In [3]:
print(df["violation_category"].value_counts(dropna=False), "\n")
print(df["violation_impact"].value_counts(), "\n")
print(df["domain_category"].value_counts(), "\n")
df["violation_name"].value_counts().head(20)

violation_category
Syntax      2634
Layout       853
Semantic      33
NaN            4
Name: count, dtype: int64 

violation_impact
serious     1376
moderate    1273
critical     475
minor        400
Name: count, dtype: int64 

domain_category
News and Media                     940
Technology Science and Research    633
Educational Platforms              575
Streaming Platforms                463
Government and Public Services     380
E-commerce                         344
Health and Wellness                165
TechnologyScienceResearch           23
Ecommerce                            1
Name: count, dtype: int64 



violation_name
color-contrast-enhanced        502
region                         382
color-contrast                 245
duplicate-id                   230
link-name                      200
landmark-unique                180
landmark-one-main              178
page-has-heading-one           177
heading-order                  164
image-alt                      115
button-name                     71
empty-heading                   66
duplicate-id-aria               65
duplicate-id-active             59
meta-viewport                   53
aria-allowed-attr               52
scrollable-region-focusable     49
html-has-lang                   48
aria-allowed-role               45
frame-title                     41
Name: count, dtype: int64

In [4]:
type_table = (
    df.groupby("violation_name")
      .agg(rows=("id", "count"), websites=("domain", "nunique"))
      .sort_values("rows", ascending=False)
)
type_table.head(25)

,rows,websites
violation_name,,
color-contrast-enhanced,502,375
region,382,299
color-contrast,245,195
duplicate-id,230,179
link-name,200,149
landmark-unique,180,135
landmark-one-main,178,156
page-has-heading-one,177,154
heading-order,164,126


In [5]:
html_dir = next(DATA.rglob("html_pages_async"))
on_disk = {p.name for p in html_dir.glob("*.html")}
expected = set(df["html_file_name"])

print("HTML folder:", html_dir)
print("files on disk:", len(on_disk))
print("files the CSV expects:", len(expected))
print("expected and present:", len(expected & on_disk))
print("expected but missing:", len(expected - on_disk))

HTML folder: ..\data\raw\accessguru\html_pages_async
files on disk: 0
files the CSV expects: 602
expected and present: 0
expected but missing: 602


In [7]:
html_files = {p.name: p for p in DATA.rglob("*.html")}
html_dir = next(iter(html_files.values())).parent
expected = set(df["html_file_name"])

print("HTML folder:", html_dir)
print("files on disk:", len(html_files))
print("files the CSV expects:", len(expected))
print("expected and present:", len(expected & set(html_files)))
print("expected but missing:", len(expected - set(html_files)))

HTML folder: ..\data\raw\accessguru\html_pages_async\workspace\FullPipeline\html_pages_async
files on disk: 564
files the CSV expects: 602
expected and present: 551
expected but missing: 51


In [6]:
row = df[df["violation_name"] == "image-alt"].iloc[0]
for col in ["web_URL", "violation_name", "violation_category", "wcag_reference",
            "violation_impact", "affected_html_elements", "supplementary_information"]:
    print(col, "=>", str(row[col])[:300])

web_URL => https://www.popsci.com
violation_name => image-alt
violation_category => Syntax
wcag_reference => ['1.1.1 Non-text Content']
violation_impact => critical
affected_html_elements => <img src="https://sync.intentiq.com/profiles_engine/ProfilesEngineServlet?at=20&amp;mi=10&amp;secure=1&amp;dpi=456059511&amp;iiqidtype=2&amp;iiqpcid=197dc4f4-beb3-45f6-b477-8ed982ddd92c&amp;iiqpciddate=1754411972204&amp;tsrnd=185_1754411972204&amp;vrref=&amp;jsver=5.082" width="1" height="1">
supplementary_information => nan
